In [0]:
for table in [
    "silver_orders",
    "silver_customers"
]:
    print(f"\n{table}:")
    spark.table(f"workspace.default.{table}").printSchema()


silver_orders:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_duration_days: double (nullable = true)
 |-- delivery_delay_days: double (nullable = true)
 |-- delivery_performance: string (nullable = true)
 |-- extreme_delay_flag: boolean (nullable = true)


silver_customers:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_zip_prefix: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

orders = spark.table("workspace.default.silver_orders")
customers = spark.table("workspace.default.silver_customers")

delivery = orders.join(
    customers.select("customer_id", "customer_state"),
    on="customer_id",
    how="left"
)

gold_delivery = (
    delivery
    .withColumn(
        "purchase_month",
        F.date_format("order_purchase_timestamp", "yyyy-MM")
    )
    .groupBy("purchase_month", "customer_state")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum(F.when(F.col("order_status") == "delivered", 1).otherwise(0)).alias("delivered_orders"),
        F.sum(F.when(F.col("delivery_performance") == "Late", 1).otherwise(0)).alias("late_orders"),
        F.sum(F.when(F.col("delivery_performance") == "Early", 1).otherwise(0)).alias("early_orders"),
        F.sum(F.when(F.col("delivery_performance") == "On Time", 1).otherwise(0)).alias("on_time_orders"),
        F.round(F.avg("delivery_duration_days"), 2).alias("avg_delivery_days"),
        F.round(F.avg("delivery_delay_days"), 2).alias("avg_delay_days"),
        F.sum(F.when(F.col("extreme_delay_flag"), 1).otherwise(0)).alias("extreme_delay_orders")
    )
    .withColumn(
        "late_delivery_rate",
        F.round(
            F.col("late_orders") / F.when(F.col("delivered_orders") > 0, F.col("delivered_orders")) * 100,
            2
        )
    )
)

(
    gold_delivery.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_delivery_performance")
)

print("Gold delivery table created:", gold_delivery.count())

Gold delivery table created: 565


In [0]:
gold = spark.table("workspace.default.gold_delivery_performance")

gold.select(
    F.sum("total_orders").alias("total_orders"),
    F.sum("delivered_orders").alias("delivered_orders"),
    F.sum("late_orders").alias("late_orders"),
    F.sum("early_orders").alias("early_orders"),
    F.sum("on_time_orders").alias("on_time_orders")
).show()

gold.orderBy(
    F.desc("late_delivery_rate")
).select(
    "purchase_month",
    "customer_state",
    "delivered_orders",
    "late_orders",
    "late_delivery_rate"
).show(10, truncate=False)

+------------+----------------+-----------+------------+--------------+
|total_orders|delivered_orders|late_orders|early_orders|on_time_orders|
+------------+----------------+-----------+------------+--------------+
|       99441|           96478|       7820|       88634|            16|
+------------+----------------+-----------+------------+--------------+

+--------------+--------------+----------------+-----------+------------------+
|purchase_month|customer_state|delivered_orders|late_orders|late_delivery_rate|
+--------------+--------------+----------------+-----------+------------------+
|2016-09       |SP            |1               |1          |100.0             |
|2017-03       |AP            |3               |2          |66.67             |
|2018-03       |SE            |17              |11         |64.71             |
|2017-11       |PI            |31              |18         |58.06             |
|2018-03       |MA            |49              |27         |55.1              |

Gold seller performance

In [0]:
for table in [
    "silver_order_items",
    "silver_sellers"
]:
    print(f"\n{table}:")
    spark.table(f"workspace.default.{table}").printSchema()


silver_order_items:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- total_item_cost: double (nullable = true)


silver_sellers:
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- seller_zip_prefix: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

items = spark.table("workspace.default.silver_order_items")
sellers = spark.table("workspace.default.silver_sellers")
orders = spark.table("workspace.default.silver_orders")

seller_sales = (
    items.groupBy("seller_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("items_sold"),
        F.round(F.sum("price"), 2).alias("product_revenue"),
        F.round(F.sum("freight_value"), 2).alias("freight_revenue")
    )
)

seller_deliveries = (
    items.select("seller_id", "order_id").distinct()
    .join(
        orders.select("order_id", "delivery_performance", "delivery_delay_days"),
        on="order_id",
        how="inner"
    )
    .groupBy("seller_id")
    .agg(
        F.countDistinct(
            F.when(F.col("delivery_performance") == "Late", F.col("order_id"))
        ).alias("late_orders"),
        F.countDistinct(
            F.when(F.col("delivery_performance") == "delivered", F.col("order_id"))
        ).alias("delivered_orders"),
        F.round(F.avg("delivery_delay_days"), 2).alias("avg_delay_days")
    )
)

gold_seller = (
    sellers
    .join(seller_sales, "seller_id", "left")
    .join(seller_deliveries, "seller_id", "left")
    .withColumn(
        "late_delivery_rate",
        F.round(
            F.col("late_orders") /
            F.when(F.col("delivered_orders") > 0, F.col("delivered_orders")) * 100,
            2
        )
    )
)

(
    gold_seller.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_seller_performance")
)

print("Gold seller table created:", gold_seller.count())

Gold seller table created: 3095


In [0]:
seller_deliveries = (
    items.select("seller_id", "order_id").distinct()
    .join(
        orders.select("order_id", "order_status", "delivery_performance", "delivery_delay_days"),
        on="order_id",
        how="inner"
    )
    .groupBy("seller_id")
    .agg(
        F.countDistinct(
            F.when(F.col("delivery_performance") == "Late", F.col("order_id"))
        ).alias("late_orders"),
        F.countDistinct(
            F.when(F.col("order_status") == "delivered", F.col("order_id"))
        ).alias("delivered_orders"),
        F.round(F.avg("delivery_delay_days"), 2).alias("avg_delay_days")
    )
)

gold_seller = (
    sellers
    .join(seller_sales, "seller_id", "left")
    .join(seller_deliveries, "seller_id", "left")
    .withColumn(
        "late_delivery_rate",
        F.round(
            F.col("late_orders") /
            F.when(F.col("delivered_orders") > 0, F.col("delivered_orders")) * 100,
            2
        )
    )
)

gold_seller.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("workspace.default.gold_seller_performance")

print("Corrected seller table:", gold_seller.count())

Corrected seller table: 3095


In [0]:
gold_seller = spark.table("workspace.default.gold_seller_performance")

gold_seller.select(
    F.count("*").alias("sellers"),
    F.sum("total_orders").alias("seller_order_count"),
    F.sum("items_sold").alias("items_sold"),
    F.sum("product_revenue").alias("product_revenue"),
    F.sum("freight_revenue").alias("freight_revenue"),
    F.sum("delivered_orders").alias("delivered_orders"),
    F.sum("late_orders").alias("late_orders")
).show()

+-------+------------------+----------+--------------------+-----------------+----------------+-----------+
|sellers|seller_order_count|items_sold|     product_revenue|  freight_revenue|delivered_orders|late_orders|
+-------+------------------+----------+--------------------+-----------------+----------------+-----------+
|   3095|            100010|    112650|1.3591643700000064E7|2251909.539999999|           97819|       7838|
+-------+------------------+----------+--------------------+-----------------+----------------+-----------+



Product category performance

In [0]:
for table in [
    "silver_products",
    "silver_category_translation"
]:
    print(f"\n{table}:")
    spark.table(f"workspace.default.{table}").printSchema()


silver_products:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- product_volume_cm3: double (nullable = true)


silver_category_translation:
root
 |-- product_category_name: string (nullable = true)
 |-- category_english: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

items = spark.table("workspace.default.silver_order_items")
products = spark.table("workspace.default.silver_products")
translations = spark.table("workspace.default.silver_category_translation")

category_sales = (
    items.join(products.select("product_id", "product_category_name"), "product_id", "left")
    .groupBy("product_category_name")
    .agg(
        F.count("*").alias("items_sold"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("price"), 2).alias("product_revenue"),
        F.round(F.sum("freight_value"), 2).alias("freight_revenue"),
        F.round(F.avg("price"), 2).alias("avg_item_price")
    )
)

gold_category = (
    category_sales
    .join(translations, "product_category_name", "left")
    .withColumn(
        "category_name",
        F.coalesce("category_english", "product_category_name", F.lit("Unknown"))
    )
    .drop("category_english")
)

(
    gold_category.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_category_performance")
)

print("Gold category table created:", gold_category.count())

Gold category table created: 74


validate the financial totals against the seller table.

In [0]:
gold_category = spark.table("workspace.default.gold_category_performance")

gold_category.select(
    F.count("*").alias("categories"),
    F.sum("items_sold").alias("items_sold"),
    F.sum("product_revenue").alias("product_revenue"),
    F.sum("freight_revenue").alias("freight_revenue")
).show()

gold_category.orderBy(
    F.desc("product_revenue")
).select(
    "category_name",
    "items_sold",
    "product_revenue",
    "freight_revenue"
).show(10, truncate=False)

+----------+----------+---------------+------------------+
|categories|items_sold|product_revenue|   freight_revenue|
+----------+----------+---------------+------------------+
|        74|    112650|   1.35916437E7|2251909.5400000014|
+----------+----------+---------------+------------------+

+---------------------+----------+---------------+---------------+
|category_name        |items_sold|product_revenue|freight_revenue|
+---------------------+----------+---------------+---------------+
|health_beauty        |9670      |1258681.34     |182566.73      |
|watches_gifts        |5991      |1205005.68     |100535.93      |
|bed_bath_table       |11115     |1036988.68     |204693.04      |
|sports_leisure       |8641      |988048.97      |168607.51      |
|computers_accessories|7827      |911954.32      |147318.08      |
|furniture_decor      |8334      |729762.49      |172749.3       |
|cool_stuff           |3796      |635290.85      |84039.1        |
|housewares           |6964      |

Payment analysis

In [0]:
spark.table("workspace.default.silver_payments").printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



gold_payment_analysis

In [0]:
from pyspark.sql import functions as F

payments = spark.table("workspace.default.silver_payments")

gold_payment = (
    payments
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("total_transactions"),
        F.countDistinct("order_id").alias("unique_orders"),
        F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
        F.round(F.avg("payment_value"), 2).alias("avg_payment_value"),
        F.round(F.avg("payment_installments"), 2).alias("avg_installments")
    )
)

(
    gold_payment.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_payment_analysis")
)

print("Gold payment table created:", gold_payment.count())

Gold payment table created: 5


In [0]:
gold_payment = spark.table("workspace.default.gold_payment_analysis")

gold_payment.orderBy(
    F.desc("total_payment_value")
).show(10, truncate=False)

+------------+------------------+-------------+-------------------+-----------------+----------------+
|payment_type|total_transactions|unique_orders|total_payment_value|avg_payment_value|avg_installments|
+------------+------------------+-------------+-------------------+-----------------+----------------+
|credit_card |76795             |76505        |1.254208419E7      |163.32           |3.51            |
|boleto      |19784             |19784        |2869361.27         |145.03           |1.0             |
|voucher     |5775              |3866         |379436.87          |65.7             |1.0             |
|debit_card  |1529              |1528         |217989.79          |142.57           |1.0             |
|not_defined |3                 |3            |0.0                |0.0              |1.0             |
+------------+------------------+-------------+-------------------+-----------------+----------------+



Customer review analysis

In [0]:
reviews = spark.table("workspace.default.silver_reviews")

gold_reviews = (
    reviews
    .groupBy("review_score", "sentiment")
    .agg(
        F.count("*").alias("total_reviews"),
        F.countDistinct("order_id").alias("unique_orders")
    )
)

(
    gold_reviews.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_review_analysis")
)

print("Gold review table created:", gold_reviews.count())

Gold review table created: 5


verify the distribution


In [0]:
gold_reviews = spark.table("workspace.default.gold_review_analysis")

gold_reviews.orderBy("review_score").show(10, truncate=False)

gold_reviews.select(
    F.sum("total_reviews").alias("total_reviews"),
    F.sum("unique_orders").alias("reviewed_orders")
).show()

+------------+---------+-------------+-------------+
|review_score|sentiment|total_reviews|unique_orders|
+------------+---------+-------------+-------------+
|1           |Negative |11424        |11393        |
|2           |Negative |3151         |3148         |
|3           |Neutral  |8179         |8160         |
|4           |Positive |19142        |19098        |
|5           |Positive |57328        |57076        |
+------------+---------+-------------+-------------+

+-------------+---------------+
|total_reviews|reviewed_orders|
+-------------+---------------+
|        99224|          98875|
+-------------+---------------+



In [0]:
orders = spark.table("workspace.default.silver_orders")
customers = spark.table("workspace.default.silver_customers")

(
    orders
    .join(customers, "customer_id")
    .groupBy("customer_city", "customer_state")
    .agg(F.countDistinct("order_id").alias("total_orders"))
    .orderBy(F.desc("total_orders"))
    .show(15, truncate=False)
)

+---------------------+--------------+------------+
|customer_city        |customer_state|total_orders|
+---------------------+--------------+------------+
|sao paulo            |SP            |15540       |
|rio de janeiro       |RJ            |6882        |
|belo horizonte       |MG            |2773        |
|brasilia             |DF            |2131        |
|curitiba             |PR            |1521        |
|campinas             |SP            |1444        |
|porto alegre         |RS            |1379        |
|salvador             |BA            |1245        |
|guarulhos            |SP            |1189        |
|sao bernardo do campo|SP            |938         |
|niteroi              |RJ            |849         |
|santo andre          |SP            |796         |
|osasco               |SP            |746         |
|santos               |SP            |713         |
|goiania              |GO            |692         |
+---------------------+--------------+------------+
only showing

Get city coordinates

In [0]:

from pyspark.sql import functions as F

customers = spark.table("workspace.default.silver_customers")
geo = spark.table("workspace.default.silver_geolocation")

customer_zip_counts = (
    customers
    .groupBy(
        "customer_city",
        "customer_state",
        "customer_zip_prefix"
    )
    .agg(F.count("*").alias("customer_count"))
)

city_locations = (
    customer_zip_counts.alias("c")
    .join(
        geo.alias("g"),
        F.col("c.customer_zip_prefix") == F.col("g.zip_prefix"),
        "inner"
    )
    .groupBy(
        "c.customer_city",
        "c.customer_state"
    )
    .agg(
        F.avg("g.latitude").alias("latitude"),
        F.avg("g.longitude").alias("longitude"),
        F.sum("c.customer_count").alias("customer_count")
    )
    .orderBy(F.desc("customer_count"))
)

city_locations.show(5, truncate=False)


+--------------+--------------+-------------------+-------------------+--------------+
|customer_city |customer_state|latitude           |longitude          |customer_count|
+--------------+--------------+-------------------+-------------------+--------------+
|sao paulo     |SP            |-23.56643001551236 |-46.633695457553806|15538         |
|rio de janeiro|RJ            |-22.903259446765272|-43.36001240506336 |6882          |
|belo horizonte|MG            |-19.898312972300882|-43.96242337994197 |2773          |
|brasilia      |DF            |-15.809934182389075|-47.96502050929183 |1960          |
|curitiba      |PR            |-25.45690107624482 |-49.27477128247801 |1521          |
+--------------+--------------+-------------------+-------------------+--------------+
only showing top 5 rows


In [0]:
from pyspark.sql import functions as F

customers = spark.table("workspace.default.silver_customers").alias("c")
geo = spark.table("workspace.default.silver_geolocation").alias("g")

city_locations = (
    customers
    .join(
        geo,
        F.col("c.customer_zip_prefix") == F.col("g.zip_prefix"),
        "inner"
    )
    .groupBy("c.customer_city", "c.customer_state")
    .agg(
        F.avg("g.latitude").alias("latitude"),
        F.avg("g.longitude").alias("longitude"),
        F.countDistinct("c.customer_id").alias("customer_count")
    )
    .orderBy(F.desc("customer_count"))
)

city_locations.show(5, truncate=False)

+--------------+--------------+-------------------+-------------------+--------------+
|customer_city |customer_state|latitude           |longitude          |customer_count|
+--------------+--------------+-------------------+-------------------+--------------+
|sao paulo     |SP            |-23.57187751528056 |-46.63344804441276 |15538         |
|rio de janeiro|RJ            |-22.922830755068002|-43.323817441148925|6882          |
|belo horizonte|MG            |-19.909021172701248|-43.956741803724384|2773          |
|brasilia      |DF            |-15.811214178596781|-47.97132650688046 |1960          |
|curitiba      |PR            |-25.45331630165661 |-49.27417299943917 |1521          |
+--------------+--------------+-------------------+-------------------+--------------+
only showing top 5 rows


Test the Open-Meteo API

In [0]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": -23.5719,
    "longitude": -46.6334,
    "start_date": "2017-01-01",
    "end_date": "2017-01-07",
    "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
    "timezone": "America/Sao_Paulo"
}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("City: Sao Paulo")
    print("Dates:", data["daily"]["time"])
    print("Temperature:", data["daily"]["temperature_2m_mean"])
    print("Rainfall:", data["daily"]["precipitation_sum"])
else:
    print(response.text[:500])

Status code: 200
City: Sao Paulo
Dates: ['2017-01-01', '2017-01-02', '2017-01-03', '2017-01-04', '2017-01-05', '2017-01-06', '2017-01-07']
Temperature: [25.1, 24.3, 23.4, 23.7, 25.0, 23.3, 23.0]
Rainfall: [2.6, 3.1, 4.3, 6.6, 1.8, 6.0, 3.1]


Gold Routing Analysis

In [0]:
from pyspark.sql import functions as F

routes = spark.table("workspace.default.silver_routes")
geo = spark.table("workspace.default.silver_geolocation")

seller_geo = geo.select(
    F.col("zip_prefix").alias("seller_zip"),
    F.col("city").alias("seller_city"),
    F.col("state").alias("seller_state")
)

customer_geo = geo.select(
    F.col("zip_prefix").alias("customer_zip"),
    F.col("city").alias("customer_city"),
    F.col("state").alias("customer_state")
)

gold_routes = (
    routes
    .join(seller_geo, "seller_zip", "left")
    .join(customer_geo, "customer_zip", "left")
    .withColumn(
        "distance_band",
        F.when(F.col("distance_km") < 100, "Short")
         .when(F.col("distance_km") < 500, "Medium")
         .when(F.col("distance_km") < 1000, "Long")
         .otherwise("Very Long")
    )
    .withColumn(
        "duration_band",
        F.when(F.col("estimated_duration_hours") < 2, "Under 2 Hours")
         .when(F.col("estimated_duration_hours") < 6, "2–6 Hours")
         .when(F.col("estimated_duration_hours") < 12, "6–12 Hours")
         .otherwise("Over 12 Hours")
    )
    .select(
        "seller_zip",
        "seller_city",
        "seller_state",
        "customer_zip",
        "customer_city",
        "customer_state",
        "shipment_count",
        "distance_km",
        "estimated_duration_hours",
        "distance_band",
        "duration_band"
    )
)

gold_routes.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_route_analysis"
)

print("Gold Routing Analysis created!")
print("Total routes:", gold_routes.count())

display(gold_routes.orderBy(F.desc("shipment_count")))

Gold Routing Analysis created!
Total routes: 50


seller_zip,seller_city,seller_state,customer_zip,customer_city,customer_state,shipment_count,distance_km,estimated_duration_hours,distance_band,duration_band
81130,curitiba,PR,80030,curitiba,PR,15,14.5,0.35,Short,Under 2 Hours
14940,ibitinga,SP,28970,araruama,RJ,15,851.75,11.18,Long,6–12 Hours
14940,ibitinga,SP,35162,ipatinga,MG,14,867.79,12.11,Long,Over 12 Hours
14940,ibitinga,SP,22793,rio de janeiro,RJ,13,752.73,9.72,Long,6–12 Hours
14940,ibitinga,SP,13212,jundiai,SP,12,288.73,3.8,Medium,2–6 Hours
14940,ibitinga,SP,22775,rio de janeiro,RJ,10,740.26,9.59,Long,6–12 Hours
14940,ibitinga,SP,11065,santos,SP,10,418.06,5.55,Medium,2–6 Hours
14940,ibitinga,SP,33200,vespasiano,MG,9,676.71,8.9,Long,6–12 Hours
14940,ibitinga,SP,29101,vila velha,ES,9,1177.47,17.29,Very Long,Over 12 Hours
14940,ibitinga,SP,13290,louveira,SP,9,275.81,3.7,Medium,2–6 Hours


Validate the Gold routing table

In [0]:
gold_routes = spark.table("workspace.default.gold_route_analysis")

gold_routes.select(
    F.count("*").alias("total_routes"),
    F.sum("shipment_count").alias("shipments_across_routes"),
    F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
    F.round(F.max("distance_km"), 2).alias("max_distance_km"),
    F.round(F.avg("estimated_duration_hours"), 2).alias("avg_duration_hours"),
    F.sum(
        F.when(
            F.col("seller_city").isNull() | F.col("customer_city").isNull(),
            1
        ).otherwise(0)
    ).alias("routes_with_missing_city")
).show()

+------------+-----------------------+---------------+---------------+------------------+------------------------+
|total_routes|shipments_across_routes|avg_distance_km|max_distance_km|avg_duration_hours|routes_with_missing_city|
+------------+-----------------------+---------------+---------------+------------------+------------------------+
|          50|                    390|          583.6|        1506.71|              7.78|                       5|
+------------+-----------------------+---------------+---------------+------------------+------------------------+



In [0]:
display(spark.sql("SHOW TABLES IN workspace.default"))

database,tableName,isTemporary
default,bronze_category_translation,false
default,bronze_customers,false
default,bronze_geolocation,false
default,bronze_order_items,false
default,bronze_orders,false
default,bronze_payments,false
default,bronze_products,false
default,bronze_reviews,false
default,bronze_routes,false
default,bronze_sellers,false


unified delivery-risk dataset

In [0]:
tables_to_check = [
    "gold_delivery_performance",
    "gold_seller_performance",
    "gold_weather_delivery_analysis",
    "gold_route_analysis"
]

for table in tables_to_check:
    print(f"\n========== {table} ==========")
    spark.table(f"workspace.default.{table}").printSchema()


========== gold_delivery_performance ==========
root
 |-- purchase_month: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- delivered_orders: long (nullable = true)
 |-- late_orders: long (nullable = true)
 |-- early_orders: long (nullable = true)
 |-- on_time_orders: long (nullable = true)
 |-- avg_delivery_days: double (nullable = true)
 |-- avg_delay_days: double (nullable = true)
 |-- extreme_delay_orders: long (nullable = true)
 |-- late_delivery_rate: double (nullable = true)


========== gold_seller_performance ==========
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- seller_zip_prefix: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- items_sold: long (nullable = true)
 |-- product_revenue: double (nullable = true)
 |-- freight_revenue: double (

Inspect order-level join keys

In [0]:
tables_to_check = [
    "silver_orders",
    "silver_customers",
    "silver_sellers",
    "silver_order_items",
    "silver_weather",
    "silver_routes"
]

for table in tables_to_check:
    print(f"\n========== {table} ==========")
    spark.table(f"workspace.default.{table}").printSchema()


========== silver_orders ==========
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_duration_days: double (nullable = true)
 |-- delivery_delay_days: double (nullable = true)
 |-- delivery_performance: string (nullable = true)
 |-- extreme_delay_flag: boolean (nullable = true)


========== silver_customers ==========
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_zip_prefix: string (

shipment-level cardinality

In [0]:
items = spark.table("workspace.default.silver_order_items")

order_seller = items.select(
    "order_id", "seller_id"
).distinct()

order_seller.groupBy("order_id").agg(
    F.countDistinct("seller_id").alias("seller_count")
).groupBy("seller_count").count().orderBy("seller_count").show()

print("Unique orders:", order_seller.select("order_id").distinct().count())
print("Unique order-seller combinations:", order_seller.count())

+------------+-----+
|seller_count|count|
+------------+-----+
|           1|97388|
|           2| 1219|
|           3|   54|
|           4|    3|
|           5|    2|
+------------+-----+

Unique orders: 98666
Unique order-seller combinations: 100010


order–seller foundation


In [0]:
orders = spark.table("workspace.default.silver_orders")
customers = spark.table("workspace.default.silver_customers")
items = spark.table("workspace.default.silver_order_items")
sellers = spark.table("workspace.default.silver_sellers")

order_seller_base = (
    items
    .groupBy("order_id", "seller_id")
    .agg(
        F.countDistinct("order_item_id").alias("items_in_shipment"),
        F.sum("price").alias("product_value"),
        F.sum("freight_value").alias("freight_value")
    )
    .join(orders, "order_id", "inner")
    .join(customers, "customer_id", "left")
    .join(sellers, "seller_id", "left")
    .select(
        "order_id",
        "seller_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "delivery_duration_days",
        "delivery_delay_days",
        "delivery_performance",
        "extreme_delay_flag",
        "customer_city",
        "customer_state",
        "customer_zip_prefix",
        "seller_city",
        "seller_state",
        "seller_zip_prefix",
        "items_in_shipment",
        "product_value",
        "freight_value"
    )
)

print("Order-seller records:", order_seller_base.count())
print("Distinct orders:", order_seller_base.select("order_id").distinct().count())

display(order_seller_base.limit(10))

Order-seller records: 100010
Distinct orders: 98666


order_id,seller_id,customer_id,order_status,order_purchase_timestamp,delivery_duration_days,delivery_delay_days,delivery_performance,extreme_delay_flag,customer_city,customer_state,customer_zip_prefix,seller_city,seller_state,seller_zip_prefix,items_in_shipment,product_value,freight_value
0009c9a17f916a706d71784483a5d643,fcb5ace8bcc92f75707dc0f01a27d269,8a250edc40ebc5c3940ebc940f16a7eb,delivered,2018-04-25T09:10:41.000Z,5.36,-8.25,Early,false,campinas,SP,13044,guarulhos,SP,07032,1,639.0,11.34
0019c29108428acffd089c36103c9440,77530e9772f57a62c906e1c21538ab82,5f6bbac628ae418db4e0f92932f899c1,delivered,2018-03-06T06:40:28.000Z,10.66,-25.06,Early,false,monte siao,MG,37580,curitiba,PR,80310,1,59.9,19.95
00276d5c3491fbf55305e26891040df9,8f78f0903005064036736c7173a5c2ed,54686c41e86854e35879efbd1c2cab1b,delivered,2018-02-13T11:47:21.000Z,31.34,-5.17,Early,false,cambe,PR,86185,tiradentes,MG,36325,1,44.9,23.22
002f19a65a2ddd70a090297872e6d64e,1835b56ce799e6a4dc4eddc053f04066,7fa80efb1ef15ca4104627910c29791c,shipped,2018-03-21T13:05:30.000Z,null,null,Not Delivered,false,camacari,BA,42804,ibitinga,SP,14940,1,53.98,23.31
00418a49a685c6bb60633291c3fab17a,6560211a19b47992c3666cc44a7e94c0,7d9edced80af5c9503cbf88a9e5f2a3e,delivered,2017-10-03T20:43:03.000Z,1.97,-13.17,Early,false,osasco,SP,06083,sao paulo,SP,05849,1,49.0,7.78
004b62108b876867938746ad3154167f,c70c1b0d8ca86052f45a432a38b73958,7ddb7e4242fcc518842aade22f5bee4d,delivered,2018-07-04T20:34:26.000Z,8.26,-5.88,Early,false,sao paulo,SP,04547,hortolandia,SP,13186,1,110.32,8.03
00602f25bffa1dcfb71e202fbf9824fb,86ccac0b835037332a596a33b6949ee1,d91dc5b08f5bb5e181a878580f4ec576,delivered,2017-11-01T17:45:52.000Z,26.14,-13.12,Early,false,belem,PA,66033,blumenau,SC,89041,1,39.9,54.02
0072847849eafe70995fa51a714c81ac,de23c3b98a88888289c6f5cc1209054a,727dc68f327d34b77009f5d3da5250d9,delivered,2018-05-14T13:44:52.000Z,8.36,-6.07,Early,false,iguape,SP,11920,sao paulo,SP,05530,1,219.99,14.9
009742b43f4530bc3db7f62332c50f4b,17e34d8224d27a541263c4c64b11a56b,84dac31647902f79570207a7f29c2675,delivered,2017-10-24T15:15:58.000Z,43.14,26.78,Late,false,santo andre,SP,09015,riberao preto,SP,14085,1,133.74,12.44
00a174dbf11967d260033fa6ab711493,31344c16881c08a8a72c6d2eb29918c1,ae06f7ef9cae527f248b5401c5663265,delivered,2017-11-27T10:29:23.000Z,22.38,-0.19,Early,false,barra mansa,RJ,27310,uba,MG,36500,1,249.9,41.53


In [0]:
order_seller_base.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.countDistinct("seller_id").alias("unique_sellers"),
    F.sum(
        F.when(F.col("customer_city").isNull(), 1).otherwise(0)
    ).alias("missing_customer_city"),
    F.sum(
        F.when(F.col("seller_city").isNull(), 1).otherwise(0)
    ).alias("missing_seller_city"),
    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("missing_order_id")
).show()

+----------+-------------+--------------+---------------------+-------------------+----------------+
|total_rows|unique_orders|unique_sellers|missing_customer_city|missing_seller_city|missing_order_id|
+----------+-------------+--------------+---------------------+-------------------+----------------+
|    100010|        98666|          3095|                    0|                  0|               0|
+----------+-------------+--------------+---------------------+-------------------+----------------+



Attach weather information


In [0]:
weather = spark.table("workspace.default.silver_weather")

weather_monthly = (
    weather
    .withColumn(
        "city_key",
        F.lower(F.regexp_replace(F.col("city"), "_", " "))
    )
    .withColumn(
        "purchase_month",
        F.date_format("weather_date", "yyyy-MM")
    )
    .groupBy("city_key", "purchase_month")
    .agg(
        F.round(F.avg("temperature_c"), 2).alias("avg_temperature_c"),
        F.round(F.sum("precipitation_mm"), 2).alias("total_rainfall_mm"),
        F.round(F.avg("max_wind_speed_kmh"), 2).alias("avg_wind_speed_kmh")
    )
)

order_seller_weather = (
    order_seller_base
    .withColumn(
        "city_key",
        F.lower(F.trim(F.col("customer_city")))
    )
    .withColumn(
        "purchase_month",
        F.date_format("order_purchase_timestamp", "yyyy-MM")
    )
    .join(
        weather_monthly,
        ["city_key", "purchase_month"],
        "left"
    )
    .drop("city_key", "purchase_month")
)

print("Rows after weather join:", order_seller_weather.count())

order_seller_weather.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(F.col("avg_temperature_c").isNotNull(), 1).otherwise(0)
    ).alias("rows_with_weather"),
    F.sum(
        F.when(F.col("avg_temperature_c").isNull(), 1).otherwise(0)
    ).alias("rows_without_weather")
).show()

display(
    order_seller_weather
    .filter(F.col("avg_temperature_c").isNotNull())
    .select(
        "order_id",
        "customer_city",
        "order_purchase_timestamp",
        "avg_temperature_c",
        "total_rainfall_mm",
        "avg_wind_speed_kmh",
        "delivery_performance"
    )
    .limit(10)
)

Rows after weather join: 100010
+----------+-----------------+--------------------+
|total_rows|rows_with_weather|rows_without_weather|
+----------+-----------------+--------------------+
|    100010|            28985|               71025|
+----------+-----------------+--------------------+



order_id,customer_city,order_purchase_timestamp,avg_temperature_c,total_rainfall_mm,avg_wind_speed_kmh,delivery_performance
01ae5e4c5d2549f7c05000a2f5856c28,rio de janeiro,2017-11-05T15:01:09.000Z,23.23,151.5,15.91,Early
03f2f876bfee9ea9a502ec0d8a664e4c,sao paulo,2017-11-29T19:35:55.000Z,19.62,104.3,22.66,Early
085bc46be56c7b3c4d5d805843d02e5e,sao paulo,2017-07-04T18:48:46.000Z,15.15,4.7,19.53,Early
08c7ad853a45735072fee2be62052da3,sao paulo,2018-06-20T22:51:17.000Z,17.35,54.0,17.69,Early
09c8e8de8792915ab68564b4420f246f,brasilia,2018-08-22T09:31:42.000Z,21.78,11.6,14.45,Early
0ced505a94fa52355ca31070c281df6e,sao paulo,2018-04-20T09:46:31.000Z,19.77,28.3,19.55,Early
10ac5d510cf3a86b9f09342e46c3972d,sao paulo,2018-01-30T17:08:30.000Z,21.74,227.0,19.57,Early
1c709af6dc28e45be0dda17b368271df,sao paulo,2018-08-21T10:20:46.000Z,15.76,57.3,19.07,Late
1d942a0debeb43eee8a316c245b9cc34,belo horizonte,2017-09-16T11:15:51.000Z,19.69,11.3,20.89,Early
1facf0f943e2e863012876581d779f16,brasilia,2017-07-04T18:59:13.000Z,17.59,0.7,20.01,Early


In [0]:
print("Original rows:", order_seller_base.count())
print("Rows after weather join:", order_seller_weather.count())

order_seller_weather.agg(
    F.count("*").alias("total_rows"),
    F.count("avg_temperature_c").alias("weather_matched"),
    F.count("total_rainfall_mm").alias("rainfall_matched"),
    F.count("avg_wind_speed_kmh").alias("wind_matched")
).show()

Original rows: 100010
Rows after weather join: 100010
+----------+---------------+----------------+------------+
|total_rows|weather_matched|rainfall_matched|wind_matched|
+----------+---------------+----------------+------------+
|    100010|          28985|           28985|       28985|
+----------+---------------+----------------+------------+



In [0]:
routes = spark.table("workspace.default.silver_routes")

order_seller_weather_routes = (
    order_seller_weather
    .join(
        routes.select(
            "seller_zip",
            "customer_zip",
            "distance_km",
            "estimated_duration_hours"
        ),
        (
            (order_seller_weather.seller_zip_prefix == routes.seller_zip) &
            (order_seller_weather.customer_zip_prefix == routes.customer_zip)
        ),
        "left"
    )
    .drop("seller_zip", "customer_zip")
)

print("Rows after routing join:", order_seller_weather_routes.count())

order_seller_weather_routes.agg(
    F.count("*").alias("total_rows"),
    F.count("distance_km").alias("routes_matched"),
    F.count("estimated_duration_hours").alias("duration_matched")
).show()

display(
    order_seller_weather_routes
    .filter(F.col("distance_km").isNotNull())
    .select(
        "order_id",
        "seller_city",
        "customer_city",
        "distance_km",
        "estimated_duration_hours",
        "delivery_performance"
    )
    .limit(10)
)

Rows after routing join: 100010
+----------+--------------+----------------+
|total_rows|routes_matched|duration_matched|
+----------+--------------+----------------+
|    100010|           369|             369|
+----------+--------------+----------------+



order_id,seller_city,customer_city,distance_km,estimated_duration_hours,delivery_performance
359b8545fa52faae433eea1788c2556d,ibitinga,ipatinga,867.79,12.11,Early
82cfc721893072e906ea29b1f0fa08f0,ibitinga,rio de janeiro,742.83,9.53,Early
8d80eb399a69c1577a7bfbc88aebe90f,ibitinga,vila velha,1177.47,17.29,Early
b5a304ee10747d3bdc90d7aef4b76a4f,ibitinga,indaiatuba,277.87,3.69,Early
c96209cd1d43d071d3bdf48d299b7aa5,curitiba,curitiba,14.5,0.35,Early
0c4284c2b4798ef2f308a6748fa63dc3,ibitinga,extrema,356.38,4.69,Early
61f6d3c81710a644622f59cd9dd666bf,piracicaba,sao jose dos campos,224.07,2.86,Early
e0940e87c03215f63faf04c6d7675616,ibitinga,araruama,851.75,11.18,Early
6d649c890e34db80ede180e9497aeb6a,ibitinga,indaiatuba,277.87,3.69,Early
38b1f1d241fb8a17915d58514c912418,ibitinga,araruama,851.75,11.18,Not Delivered


In [0]:
print("Original rows:", order_seller_weather.count())
print("Rows after routing join:", order_seller_weather_routes.count())

order_seller_weather_routes.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.count("distance_km").alias("routes_matched"),
    F.count("estimated_duration_hours").alias("duration_matched")
).show()

Original rows: 100010
Rows after routing join: 100010
+----------+-------------+--------------+----------------+
|total_rows|unique_orders|routes_matched|duration_matched|
+----------+-------------+--------------+----------------+
|    100010|        98666|           369|             369|
+----------+-------------+--------------+----------------+



In [0]:
seller_performance = spark.table("workspace.default.gold_seller_performance")

order_seller_enriched = (
    order_seller_weather_routes
    .join(
        seller_performance.select(
            "seller_id",
            F.col("late_delivery_rate").alias("seller_late_delivery_rate"),
            F.col("avg_delay_days").alias("seller_avg_delay_days")
        ),
        "seller_id",
        "left"
    )
)

print("Rows after seller join:", order_seller_enriched.count())

order_seller_enriched.select(
    "order_id",
    "seller_id",
    "seller_late_delivery_rate",
    "seller_avg_delay_days",
    "delivery_performance"
).show(10, truncate=False)

Rows after seller join: 100010
+--------------------------------+--------------------------------+-------------------------+---------------------+--------------------+
|order_id                        |seller_id                       |seller_late_delivery_rate|seller_avg_delay_days|delivery_performance|
+--------------------------------+--------------------------------+-------------------------+---------------------+--------------------+
|0009c9a17f916a706d71784483a5d643|fcb5ace8bcc92f75707dc0f01a27d269|3.39                     |-8.65                |Early               |
|0019c29108428acffd089c36103c9440|77530e9772f57a62c906e1c21538ab82|11.54                    |-10.25               |Early               |
|00276d5c3491fbf55305e26891040df9|8f78f0903005064036736c7173a5c2ed|7.69                     |-11.17               |Early               |
|002f19a65a2ddd70a090297872e6d64e|1835b56ce799e6a4dc4eddc053f04066|11.75                    |-9.62                |Not Delivered       |
|00418a49a

In [0]:
order_seller_enriched.select(
    F.count("*").alias("total_records"),
    F.count("seller_late_delivery_rate").alias("seller_metrics_available"),
    F.count("avg_temperature_c").alias("weather_available"),
    F.count("distance_km").alias("routing_available"),
    F.min("seller_late_delivery_rate").alias("min_seller_late_rate"),
    F.max("seller_late_delivery_rate").alias("max_seller_late_rate"),
    F.min("seller_avg_delay_days").alias("min_seller_avg_delay"),
    F.max("seller_avg_delay_days").alias("max_seller_avg_delay"),
    F.min("distance_km").alias("min_route_km"),
    F.max("distance_km").alias("max_route_km")
).show(truncate=False)

+-------------+------------------------+-----------------+-----------------+--------------------+--------------------+--------------------+--------------------+------------+------------+
|total_records|seller_metrics_available|weather_available|routing_available|min_seller_late_rate|max_seller_late_rate|min_seller_avg_delay|max_seller_avg_delay|min_route_km|max_route_km|
+-------------+------------------------+-----------------+-----------------+--------------------+--------------------+--------------------+--------------------+------------+------------+
|100010       |99844                   |28985            |369              |0.0                 |100.0               |-65.32              |167.71              |14.5        |1506.71     |
+-------------+------------------------+-----------------+-----------------+--------------------+--------------------+--------------------+--------------------+------------+------------+



In [0]:
seller_performance.groupBy(
    F.when(F.col("total_orders") < 5, "1–4 orders")
     .when(F.col("total_orders") < 20, "5–19 orders")
     .when(F.col("total_orders") < 50, "20–49 orders")
     .otherwise("50+ orders")
     .alias("seller_volume_band")
).agg(
    F.count("*").alias("seller_count"),
    F.round(F.avg("late_delivery_rate"), 2).alias("avg_late_rate")
).orderBy("seller_volume_band").show()

+------------------+------------+-------------+
|seller_volume_band|seller_count|avg_late_rate|
+------------------+------------+-------------+
|        1–4 orders|        1301|         8.91|
|      20–49 orders|         389|         7.79|
|        50+ orders|         429|          7.9|
|       5–19 orders|         976|         8.28|
+------------------+------------+-------------+



In [0]:
low_volume_sellers = (
    seller_performance
    .filter(F.col("total_orders") < 5)
    .select("seller_id")
    .withColumn("is_low_volume", F.lit(True))
)

order_seller_enriched = (
    order_seller_enriched
    .join(low_volume_sellers, "seller_id", "left")
    .withColumn(
        "seller_reliability",
        F.when(F.col("seller_late_delivery_rate").isNull(), "Unavailable")
         .when(F.col("is_low_volume") == True, "Low Sample")
         .otherwise("Sufficient Sample")
    )
    .drop("is_low_volume")
)

order_seller_enriched.groupBy("seller_reliability").count().show()

+------------------+-----+
|seller_reliability|count|
+------------------+-----+
| Sufficient Sample|97438|
|        Low Sample| 2406|
|       Unavailable|  166|
+------------------+-----+



In [0]:
order_seller_enriched.select(
    F.min("total_rainfall_mm").alias("min_monthly_rainfall"),
    F.max("total_rainfall_mm").alias("max_monthly_rainfall"),
    F.min("avg_wind_speed_kmh").alias("min_avg_wind"),
    F.max("avg_wind_speed_kmh").alias("max_avg_wind"),
    F.min("avg_temperature_c").alias("min_temperature"),
    F.max("avg_temperature_c").alias("max_temperature")
).show(truncate=False)

+--------------------+--------------------+------------+------------+---------------+---------------+
|min_monthly_rainfall|max_monthly_rainfall|min_avg_wind|max_avg_wind|min_temperature|max_temperature|
+--------------------+--------------------+------------+------------+---------------+---------------+
|0.0                 |365.5               |10.96       |22.66       |13.22          |26.63          |
+--------------------+--------------------+------------+------------+---------------+---------------+



In [0]:
order_seller_enriched = (
    order_seller_enriched
    .withColumn(
        "rainfall_band",
        F.when(F.col("total_rainfall_mm").isNull(), "Unavailable")
         .when(F.col("total_rainfall_mm") < 50, "Low")
         .when(F.col("total_rainfall_mm") <= 150, "Moderate")
         .otherwise("High")
    )
    .withColumn(
        "wind_band",
        F.when(F.col("avg_wind_speed_kmh").isNull(), "Unavailable")
         .when(F.col("avg_wind_speed_kmh") < 15, "Low")
         .when(F.col("avg_wind_speed_kmh") <= 20, "Moderate")
         .otherwise("High")
    )
    .withColumn(
        "route_distance_band",
        F.when(F.col("distance_km").isNull(), "Unavailable")
         .when(F.col("distance_km") < 100, "Short")
         .when(F.col("distance_km") < 500, "Medium")
         .when(F.col("distance_km") < 1000, "Long")
         .otherwise("Very Long")
    )
)

order_seller_enriched.groupBy("rainfall_band").count().show()
order_seller_enriched.groupBy("wind_band").count().show()
order_seller_enriched.groupBy("route_distance_band").count().show()

+-------------+-----+
|rainfall_band|count|
+-------------+-----+
|         High| 5583|
|  Unavailable|71025|
|     Moderate|13609|
|          Low| 9793|
+-------------+-----+

+-----------+-----+
|  wind_band|count|
+-----------+-----+
|   Moderate|13138|
|Unavailable|71025|
|       High| 7322|
|        Low| 8525|
+-----------+-----+

+-------------------+-----+
|route_distance_band|count|
+-------------------+-----+
|        Unavailable|99641|
|               Long|  209|
|          Very Long|   24|
|             Medium|  115|
|              Short|   21|
+-------------------+-----+



In [0]:
(
    order_seller_enriched
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_delivery_risk_observatory")
)

print("Gold delivery risk observatory created successfully.")

spark.table(
    "workspace.default.gold_delivery_risk_observatory"
).printSchema()

Gold delivery risk observatory created successfully.
root
 |-- seller_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- delivery_duration_days: double (nullable = true)
 |-- delivery_delay_days: double (nullable = true)
 |-- delivery_performance: string (nullable = true)
 |-- extreme_delay_flag: boolean (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_zip_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- seller_zip_prefix: string (nullable = true)
 |-- items_in_shipment: long (nullable = true)
 |-- product_value: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- avg_temperature_c: double (nullable = true)
 |-- total_rainfall_mm: double (nullable = 

In [0]:
gold_risk = spark.table(
    "workspace.default.gold_delivery_risk_observatory"
)

gold_risk.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.countDistinct(
        F.concat_ws("_", "order_id", "seller_id")
    ).alias("unique_order_seller_pairs"),
    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("missing_order_ids"),
    F.sum(
        F.when(F.col("seller_id").isNull(), 1).otherwise(0)
    ).alias("missing_seller_ids")
).show()

+-------------+-------------+-------------------------+-----------------+------------------+
|total_records|unique_orders|unique_order_seller_pairs|missing_order_ids|missing_seller_ids|
+-------------+-------------+-------------------------+-----------------+------------------+
|       100010|        98666|                   100010|                0|                 0|
+-------------+-------------+-------------------------+-----------------+------------------+



In [0]:
gold_risk.agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum(
        F.when(F.col("order_status") == "delivered", 1).otherwise(0)
    ).alias("delivered_order_seller_records"),
    F.sum(
        F.when(F.col("delivery_performance") == "Late", 1).otherwise(0)
    ).alias("late_order_seller_records"),
    F.round(F.avg("delivery_duration_days"), 2).alias("avg_delivery_days"),
    F.round(F.avg("delivery_delay_days"), 2).alias("avg_delivery_delay_days"),
    F.round(F.avg("product_value"), 2).alias("avg_shipment_product_value"),
    F.count("avg_temperature_c").alias("weather_covered_records"),
    F.count("distance_km").alias("route_covered_records")
).show(truncate=False)

+------------+------------------------------+-------------------------+-----------------+-----------------------+--------------------------+-----------------------+---------------------+
|total_orders|delivered_order_seller_records|late_order_seller_records|avg_delivery_days|avg_delivery_delay_days|avg_shipment_product_value|weather_covered_records|route_covered_records|
+------------+------------------------------+-------------------------+-----------------+-----------------------+--------------------------+-----------------------+---------------------+
|98666       |97819                         |7838                     |12.51            |-11.25                 |135.9                     |28985                  |369                  |
+------------+------------------------------+-------------------------+-----------------+-----------------------+--------------------------+-----------------------+---------------------+



KPI Gold table

In [0]:
kpi_data = gold_risk.agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.count("*").alias("total_order_seller_records"),
    F.sum(
        F.when(F.col("order_status") == "delivered", 1).otherwise(0)
    ).alias("delivered_order_seller_records"),
    F.sum(
        F.when(F.col("delivery_performance") == "Late", 1).otherwise(0)
    ).alias("late_order_seller_records"),
    F.round(F.avg("delivery_duration_days"), 2).alias("avg_delivery_days"),
    F.round(F.avg("delivery_delay_days"), 2).alias("avg_delivery_variance_days"),
    F.count("avg_temperature_c").alias("weather_covered_records"),
    F.count("distance_km").alias("route_covered_records")
)

kpi_data.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_logistics_kpis"
)

display(kpi_data)

total_orders,total_order_seller_records,delivered_order_seller_records,late_order_seller_records,avg_delivery_days,avg_delivery_variance_days,weather_covered_records,route_covered_records
98666,100010,97819,7838,12.51,-11.25,28985,369


In [0]:
display(
    gold_risk
    .groupBy("customer_state")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum(
            F.when(F.col("delivery_performance") == "Late", 1).otherwise(0)
        ).alias("late_order_seller_records"),
        F.sum(
            F.when(F.col("delivery_performance").isin(
                "Late", "Early", "On Time"
            ), 1).otherwise(0)
        ).alias("delivered_order_seller_records"),
        F.round(
            F.avg(
                F.when(F.col("delivery_performance").isin(
                    "Late", "Early", "On Time"
                ),
                F.when(F.col("delivery_performance") == "Late", 1.0).otherwise(0.0))
            ) * 100, 2
        ).alias("late_rate_pct")
    )
    .orderBy(F.desc("late_rate_pct"))
)

customer_state,total_orders,late_order_seller_records,delivered_order_seller_records,late_rate_pct
AL,411,95,399,23.81
MA,740,142,729,19.48
PI,493,76,478,15.9
CE,1327,196,1290,15.19
SE,345,51,338,15.09
BA,3358,456,3294,13.84
RJ,12762,1669,12515,13.34
TO,279,35,282,12.41
PA,970,118,955,12.36
RR,46,5,41,12.2


In [0]:
state_performance = (
    gold_risk
    .groupBy("order_id", "customer_state")
    .agg(
        F.max(
            F.when(F.col("delivery_performance") == "Late", 1).otherwise(0)
        ).alias("is_late"),
        F.max(
            F.when(
                F.col("delivery_performance").isin("Late", "Early", "On Time"),
                1
            ).otherwise(0)
        ).alias("is_delivered")
    )
    .groupBy("customer_state")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("is_late").alias("late_orders"),
        F.sum("is_delivered").alias("delivered_orders")
    )
    .withColumn(
        "late_delivery_rate_pct",
        F.round(
            F.col("late_orders") / F.col("delivered_orders") * 100, 2
        )
    )
)

display(state_performance.orderBy(F.desc("late_delivery_rate_pct")))

customer_state,total_orders,late_orders,delivered_orders,late_delivery_rate_pct
AL,411,95,397,23.93
MA,740,141,717,19.67
PI,493,76,476,15.97
CE,1327,196,1279,15.32
SE,345,51,335,15.22
BA,3358,456,3256,14.0
RJ,12762,1663,12350,13.47
TO,279,35,274,12.77
PA,970,117,946,12.37
ES,2025,244,1995,12.23


In [0]:
(
    state_performance
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_state_delivery_performance")
)

print("State delivery performance table created.")

display(
    spark.table("workspace.default.gold_state_delivery_performance")
    .orderBy(F.desc("late_delivery_rate_pct"))
)

State delivery performance table created.


customer_state,total_orders,late_orders,delivered_orders,late_delivery_rate_pct
AL,411,95,397,23.93
MA,740,141,717,19.67
PI,493,76,476,15.97
CE,1327,196,1279,15.32
SE,345,51,335,15.22
BA,3358,456,3256,14.0
RJ,12762,1663,12350,13.47
TO,279,35,274,12.77
PA,970,117,946,12.37
ES,2025,244,1995,12.23


In [0]:
monthly_performance = (
    gold_risk
    .withColumn(
        "purchase_month",
        F.date_format("order_purchase_timestamp", "yyyy-MM")
    )
    .groupBy("purchase_month", "order_id")
    .agg(
        F.max(
            F.when(F.col("delivery_performance") == "Late", 1).otherwise(0)
        ).alias("is_late"),
        F.max(
            F.when(
                F.col("delivery_performance").isin("Late", "Early", "On Time"),
                1
            ).otherwise(0)
        ).alias("is_delivered"),
        F.max("delivery_duration_days").alias("delivery_duration_days")
    )
    .groupBy("purchase_month")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("is_late").alias("late_orders"),
        F.sum("is_delivered").alias("delivered_orders"),
        F.round(F.avg("delivery_duration_days"), 2).alias("avg_delivery_days")
    )
    .withColumn(
        "late_delivery_rate_pct",
        F.round(
            F.try_divide(F.col("late_orders") * 100, F.col("delivered_orders")),
            2
        )
    )
    .orderBy("purchase_month")
)

display(monthly_performance)

purchase_month,total_orders,late_orders,delivered_orders,avg_delivery_days,late_delivery_rate_pct
2016-09,3,1,1,54.81,100.0
2016-10,308,3,265,19.6,1.13
2016-12,1,0,1,4.69,0.0
2017-01,789,23,750,12.65,3.07
2017-02,1733,53,1653,13.17,3.21
2017-03,2641,142,2546,12.95,5.58
2017-04,2391,181,2303,14.92,7.86
2017-05,3660,128,3545,11.32,3.61
2017-06,3217,121,3135,12.01,3.86
2017-07,3969,133,3872,11.59,3.43


In [0]:
(
    monthly_performance
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_monthly_delivery_performance")
)

print("Monthly delivery performance table created.")

Monthly delivery performance table created.


In [0]:

weather_analysis = spark.table(
    "workspace.default.gold_weather_delivery_analysis"
)

display(
    weather_analysis
    .filter(F.col("delivered_orders") >= 100)
    .orderBy(F.desc("late_delivery_rate"))
    .select(
        "city_key",
        "month",
        "delivered_orders",
        "late_orders",
        "late_delivery_rate",
        "avg_temperature_c",
        "total_rainfall_mm",
        "avg_wind_speed_kmh"
    )
)


city_key,month,delivered_orders,late_orders,late_delivery_rate,avg_temperature_c,total_rainfall_mm,avg_wind_speed_kmh
rio de janeiro,2018-02,491,184,37.47,24.36,225.4,11.5
rio de janeiro,2017-11,556,147,26.44,23.23,151.5,15.91
belo horizonte,2018-03,217,56,25.81,22.24,174.8,13.45
rio de janeiro,2017-12,386,90,23.32,24.49,131.7,13.71
rio de janeiro,2018-03,408,85,20.83,25.14,143.3,11.34
brasilia,2018-03,147,30,20.41,21.92,113.3,12.52
rio de janeiro,2018-01,470,87,18.51,25.39,191.8,11.64
sao paulo,2018-08,1269,229,18.05,15.76,57.3,19.07
curitiba,2018-03,110,18,16.36,20.84,154.5,16.37
brasilia,2018-02,170,25,14.71,21.46,212.7,13.81


In [0]:
weather_analysis.printSchema()

root
 |-- city_key: string (nullable = true)
 |-- month: string (nullable = true)
 |-- delivered_orders: long (nullable = true)
 |-- late_orders: long (nullable = true)
 |-- avg_delivery_days: double (nullable = true)
 |-- late_delivery_rate: double (nullable = true)
 |-- avg_temperature_c: double (nullable = true)
 |-- total_rainfall_mm: double (nullable = true)
 |-- avg_wind_speed_kmh: double (nullable = true)



In [0]:
display(
    weather_analysis
    .filter(F.col("delivered_orders") >= 100)
    .orderBy(F.desc("late_delivery_rate"))
    .select(
        "city_key",
        "month",
        "delivered_orders",
        "late_orders",
        F.round(F.col("late_delivery_rate") * 100, 2).alias("late_rate_pct"),
        "avg_temperature_c",
        "total_rainfall_mm",
        "avg_wind_speed_kmh"
    )
)

city_key,month,delivered_orders,late_orders,late_rate_pct,avg_temperature_c,total_rainfall_mm,avg_wind_speed_kmh
rio de janeiro,2018-02,491,184,3747.0,24.36,225.4,11.5
rio de janeiro,2017-11,556,147,2644.0,23.23,151.5,15.91
belo horizonte,2018-03,217,56,2581.0,22.24,174.8,13.45
rio de janeiro,2017-12,386,90,2332.0,24.49,131.7,13.71
rio de janeiro,2018-03,408,85,2083.0,25.14,143.3,11.34
brasilia,2018-03,147,30,2041.0,21.92,113.3,12.52
rio de janeiro,2018-01,470,87,1851.0,25.39,191.8,11.64
sao paulo,2018-08,1269,229,1805.0,15.76,57.3,19.07
curitiba,2018-03,110,18,1636.0,20.84,154.5,16.37
brasilia,2018-02,170,25,1471.0,21.46,212.7,13.81


In [0]:
display(
    weather_analysis
    .filter(F.col("delivered_orders") >= 100)
    .orderBy(F.desc("late_delivery_rate"))
    .select(
        "city_key",
        "month",
        "delivered_orders",
        "late_orders",
        F.col("late_delivery_rate").alias("late_rate_pct"),
        "avg_temperature_c",
        "total_rainfall_mm",
        "avg_wind_speed_kmh"
    )
)

city_key,month,delivered_orders,late_orders,late_rate_pct,avg_temperature_c,total_rainfall_mm,avg_wind_speed_kmh
rio de janeiro,2018-02,491,184,37.47,24.36,225.4,11.5
rio de janeiro,2017-11,556,147,26.44,23.23,151.5,15.91
belo horizonte,2018-03,217,56,25.81,22.24,174.8,13.45
rio de janeiro,2017-12,386,90,23.32,24.49,131.7,13.71
rio de janeiro,2018-03,408,85,20.83,25.14,143.3,11.34
brasilia,2018-03,147,30,20.41,21.92,113.3,12.52
rio de janeiro,2018-01,470,87,18.51,25.39,191.8,11.64
sao paulo,2018-08,1269,229,18.05,15.76,57.3,19.07
curitiba,2018-03,110,18,16.36,20.84,154.5,16.37
brasilia,2018-02,170,25,14.71,21.46,212.7,13.81


In [0]:
(
    weather_analysis
    .withColumn(
        "late_rate_pct",
        F.round(F.col("late_delivery_rate"), 2)
    )
    .drop("late_delivery_rate")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_weather_dashboard")
)

print("Weather dashboard table created.")

Weather dashboard table created.


In [0]:
weather_dashboard = spark.table("workspace.default.gold_weather_dashboard")

print("Rows:", weather_dashboard.count())
weather_dashboard.printSchema()

display(weather_dashboard.orderBy(F.desc("late_rate_pct")).limit(10))

Rows: 106
root
 |-- city_key: string (nullable = true)
 |-- month: string (nullable = true)
 |-- delivered_orders: long (nullable = true)
 |-- late_orders: long (nullable = true)
 |-- avg_delivery_days: double (nullable = true)
 |-- avg_temperature_c: double (nullable = true)
 |-- total_rainfall_mm: double (nullable = true)
 |-- avg_wind_speed_kmh: double (nullable = true)
 |-- late_rate_pct: double (nullable = true)



city_key,month,delivered_orders,late_orders,avg_delivery_days,avg_temperature_c,total_rainfall_mm,avg_wind_speed_kmh,late_rate_pct
rio de janeiro,2018-02,491,184,23.86,24.36,225.4,11.5,37.47
rio de janeiro,2017-11,556,147,17.67,23.23,151.5,15.91,26.44
belo horizonte,2018-03,217,56,14.95,22.24,174.8,13.45,25.81
rio de janeiro,2017-12,386,90,20.31,24.49,131.7,13.71,23.32
rio de janeiro,2018-03,408,85,22.47,25.14,143.3,11.34,20.83
brasilia,2018-03,147,30,16.28,21.92,113.3,12.52,20.41
rio de janeiro,2018-01,470,87,18.63,25.39,191.8,11.64,18.51
sao paulo,2018-08,1269,229,5.75,15.76,57.3,19.07,18.05
curitiba,2018-03,110,18,13.94,20.84,154.5,16.37,16.36
brasilia,2017-01,13,2,21.48,22.63,142.3,12.83,15.38


In [0]:
gold_tables = [
    "gold_delivery_performance",
    "gold_seller_performance",
    "gold_category_performance",
    "gold_payment_analysis",
    "gold_review_analysis",
    "gold_weather_dashboard",
    "gold_route_analysis",
    "gold_delivery_risk_observatory",
    "gold_logistics_kpis",
    "gold_state_delivery_performance",
    "gold_monthly_delivery_performance"
]

for table in gold_tables:
    df = spark.table(f"workspace.default.{table}")
    print(f"{table}: {df.count()} rows")

gold_delivery_performance: 565 rows
gold_seller_performance: 3095 rows
gold_category_performance: 74 rows
gold_payment_analysis: 5 rows
gold_review_analysis: 5 rows
gold_weather_dashboard: 106 rows
gold_route_analysis: 50 rows
gold_delivery_risk_observatory: 100010 rows
gold_logistics_kpis: 1 rows
gold_state_delivery_performance: 27 rows
gold_monthly_delivery_performance: 24 rows
